In [ ]:
from src.load_data import load_transactions, load_cards, load_users, load_mcc_codes

In [ ]:
df_transaction = load_transactions()
df_card = load_cards()
df_users = load_users()
df_mcc = load_mcc_codes()

for name, df in [("transactions", df_transaction), ("cards", df_card),
                  ("users", df_users), ("mcc", df_mcc)]:
    print(f"{name}: {df.shape[0]:,} lignes, {df.shape[1]} colonnes, "
          f"{df.memory_usage(deep=True).sum()/1e6:.1f} MB")

In [ ]:
from src.clean_data import convert_currency_column, report_conversion_issues

# transactions_data
original = df_transaction["amount"].copy()
df_transaction["amount"] = convert_currency_column(df_transaction["amount"])
report_conversion_issues(original, df_transaction["amount"], "transactions.amount")

# cards_data
original = df_card["credit_limit"].copy()
df_card["credit_limit"] = convert_currency_column(df_card["credit_limit"])
report_conversion_issues(original, df_card["credit_limit"], "cards.credit_limit")

# users_data
for col in ["per_capita_income", "yearly_income", "total_debt"]:
    original = df_users[col].copy()
    df_users[col] = convert_currency_column(df_users[col])
    report_conversion_issues(original, df_users[col], f"users.{col}")

In [ ]:
df_transaction.sample(5)   # 5 lignes au hasard

In [ ]:
from src.clean_data import (
    convert_month_year_column, check_datetime_column, report_conversion_issues
)

# cards_data
original = df_card["expires"].copy()
df_card["expires"] = convert_month_year_column(df_card["expires"], end_of_month=True)
report_conversion_issues(original, df_card["expires"], "cards.expires")

original = df_card["acct_open_date"].copy()
df_card["acct_open_date"] = convert_month_year_column(df_card["acct_open_date"])
report_conversion_issues(original, df_card["acct_open_date"], "cards.acct_open_date")

# Vérifications
check_datetime_column(df_card["expires"], "cards.expires")
check_datetime_column(df_card["acct_open_date"], "cards.acct_open_date")
check_datetime_column(df_transaction["date"], "transactions.date")

In [ ]:
# 1. Carte expirée avant l'ouverture du compte (attendu : 0)
print((df_card["expires"] < df_card["acct_open_date"]).sum())

# 2. Cartes ouvertes après la dernière transaction
print((df_card["acct_open_date"] > df_transaction["date"].max()).sum())

# 3. Cartes expirées avant la première transaction
print((df_card["expires"] < df_transaction["date"].min()).sum())

In [ ]:
import pandas as pd

# Première et dernière transaction de chaque carte
tx = df_transaction.groupby("card_id")["date"].agg(first_tx="min", last_tx="max")
cards = df_card.set_index("id").join(tx)

# Cartes sans aucune transaction
print("Sans transaction :", cards["first_tx"].isna().sum())

# Cartes avec une transaction AVANT leur mois d'ouverture
print("Transaction avant ouverture :", (cards["first_tx"] < cards["acct_open_date"]).sum())

# Cartes avec une transaction APRÈS leur date d'expiration (fin de mois incluse)
print("Transaction après expiration :",
      (cards["last_tx"] >= cards["expires"] + pd.Timedelta(days=1)).sum())

# Parmi les 1189 cartes ouvertes après le 31 oct. 2019
late = cards["acct_open_date"] > df_transaction["date"].max()
print("Ouvertes tard, mais avec transactions :", (late & cards["first_tx"].notna()).sum())

# Parmi les 211 cartes expirées avant 2010
old = cards["expires"] < df_transaction["date"].min()
print("Expirées avant 2010, mais avec transactions :", (old & cards["first_tx"].notna()).sum())

In [ ]:
avant = cards[cards["first_tx"] < cards["acct_open_date"]]
print((avant["acct_open_date"] - avant["first_tx"]).dt.days.describe())
print(avant.head())

apres = cards[cards["last_tx"] >= cards["expires"] + pd.Timedelta(days=1)]
print((apres["last_tx"] - apres["expires"]).dt.days.describe())
print(apres.head())

In [ ]:
from src.clean_data import build_mcc_table

df_mcc = build_mcc_table(df_mcc)
print(df_mcc.shape)
print(df_mcc.dtypes)
df_mcc.head(10)

In [ ]:
from src.clean_data import check_mcc_join

ok = check_mcc_join(df_transaction, df_mcc)
print("Jointure sans perte :", ok)

In [ ]:
from src.clean_data import build_mcc_table, add_mcc_families, check_mcc_families

df_mcc = build_mcc_table(df_mcc)
df_mcc = add_mcc_families(df_mcc)

ok = check_mcc_families(df_mcc, df_transaction)
print("Table de correspondance valide :", ok)
df_mcc.head(10)

In [ ]:
from src.clean_data import ALL_ERROR_TYPES, US_STATES_AND_TERRITORIES

# 1. Types d'erreur réellement présents
tokens = (df_transaction["errors"].dropna().astype(str)
          .str.split(",").explode().str.strip().value_counts())
print(tokens)
print("Absents de la liste :", sorted(set(tokens.index) - set(ALL_ERROR_TYPES)))

# 2. Valeurs de merchant_state qui ne sont pas des codes américains
print(sorted(set(df_transaction["merchant_state"].dropna().unique()) - US_STATES_AND_TERRITORIES))

# 3. Désaccord entre la ville ONLINE et le canal en ligne
print(pd.crosstab(df_transaction["merchant_city"].eq("ONLINE"),
                  df_transaction["use_chip"].eq("Online Transaction")))

In [ ]:
aa = df_transaction["merchant_state"].eq("AA")
print(aa.sum())
print(df_transaction.loc[aa, ["merchant_city", "zip"]].value_counts().head(10))

# Quels codes de territoires apparaissent réellement ?
print(sorted(set(df_transaction["merchant_state"].dropna().unique())
             & {"PR", "GU", "VI", "AS", "MP", "DC"}))

In [ ]:
desacc = df_transaction["merchant_city"].eq("ONLINE") & ~df_transaction["use_chip"].eq("Online Transaction")
print(df_transaction.loc[desacc, "use_chip"].value_counts())
print(df_transaction.loc[desacc, ["merchant_state", "zip"]].isna().mean())

In [ ]:
from src.clean_data import (transform_errors_column, check_errors_transformation,
                            structurer_lieu_transactions, check_location_structuration)

df_transaction = transform_errors_column(df_transaction)
check_errors_transformation(df_transaction)

df_transaction = structurer_lieu_transactions(df_transaction)
check_location_structuration(df_transaction)

In [ ]:
print(int(df_transaction["is_online"].sum()))
aa = df_transaction["merchant_state"].eq("AA")
print(df_transaction.loc[aa, ["merchant_country", "is_foreign"]].value_counts())